In [1]:
! pip install torchvision

In [2]:
! pip install transformers

In [3]:
import torch
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt

In [4]:
trans_train = transforms.Compose([transforms.Resize(75),
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(75),
                                  transforms.RandomHorizontalFlip(),
                                  transforms. ToTensor()])

trans_valid = transforms.Compose([transforms.Resize(75),
                                  transforms.CenterCrop(75),
                                   transforms.ToTensor()])

data_dir = "D:\Machine Learning\Datasets\Datasets\CatsVsDogs"
train_dataset = datasets.ImageFolder(data_dir + '/train', transform=trans_train)
test_dataset = datasets.ImageFolder(data_dir + '/val', transform=trans_valid)

batch_size = 1
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torchvision.datasets.folder.ImageFolder

In [5]:
class CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=2): # 3 for RGB, 2 for Cats vs Dogs
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=16, 
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32, 
            kernel_size=3,
            stride=1,
            padding=1
        )
        
        # After 75x75 -> Pool1 -> 37x37 -> Pool2 -> 18x18
        # Flattened size = out_channels (32) * height (18) * width (18)
        self.fc1 = nn.Linear(32 * 18 * 18, 150)
        self.fc2 = nn.Linear(150, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        # Flatten
        x = x.reshape(x.shape[0], -1) 
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

Setting the device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
device

device(type='cpu')

Hyper-Parameters

In [8]:
# Updated Hyper-parameters
in_channels = 3    # Cats & Dogs are RGB color images
num_classes = 2     # Cat or Dog (Binary classification)
learning_rate = 0.001
num_epochs = 5      # 3 is fine for a test, but 5-10 is better for actual learning

Processing of a single batch

In [ ]:
model = CNN(in_channels=in_channels, num_classes=num_classes)
train_features, train_labels = next(iter(train_loader))
model(train_features)

tensor([[-0.0591,  0.0623]], grad_fn=<AddmmBackward0>)

: 

In [ ]:
# Display image and label.

print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
# Move dimension 0 (channels) to the end: (3, 75, 75) -> (75, 75, 3)
plt.imshow(img.permute(1, 2, 0))
plt.show()
print(f"Label: {label}")

Feature batch shape: torch.Size([1, 3, 75, 75])
Labels batch shape: torch.Size([1])


All the batches

In [ ]:
batch_size = 16
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

Initialize the network

In [ ]:
model = CNN(in_channels=in_channels, num_classes=num_classes).to(device)

Loss and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Train Network

In [ ]:
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)

        # forward
        scores = model(data)
        loss = criterion(scores, targets)

        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()

100%|██████████| 18/18 [00:01<00:00, 11.41it/s]


##### Check accuracy on training & test to see how good our model

In [ ]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()
    return num_correct / num_samples


In [ ]:
print(f"Accuracy on training set: {check_accuracy(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model)*100:.2f}")

Accuracy on training set: 65.09
Accuracy on test set: 65.71
